# Commodity Buy Classification

Thin orchestration notebook. The pipeline logic lives in `src/`; this notebook wires the steps together.

Run from the repo root so `src` is importable, and place the yearly Excel workbooks in `data/raw/`.

In [ ]:
#Import libraries
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from src.data_preprocessing import build_dataset
from src.priority_system import add_priority_scores
from src.models import data_split, logistic, catboost_model, xgboost_model, autogluon_model
from src.models.evaluation import report_performance

In [ ]:
#Import datasets, clean columns, and combine all years
combined_data = build_dataset('../data/raw')

In [ ]:
# Priority Functions -> total priority score -> buy decision
combined_data = add_priority_scores(combined_data)
display(combined_data)

## Logistic Regression (base features)

In [ ]:
#Spliting dataset
X, y = data_split.make_xy(combined_data, use_priority_features=False)
X_train, X_valid, X_test, y_train, y_valid, y_test = data_split.split_train_valid_test(X, y)

In [ ]:
#HyperParamter Tuning with C
df_outcome = logistic.sweep_c(X_train, y_train, X_test, y_test)
df_outcome

In [ ]:
#Training
c_val = logistic.best_c(df_outcome)
logisticRegr = logistic.train(X_train, y_train, c_val)

In [ ]:
#Predicting
prediction = logistic.summarize_predictions(logisticRegr, X_test, y_test)

In [ ]:
#Plot confusion matrix and performance on the test set
report_performance(logisticRegr, y_test, prediction, X_test, y_test,
                   title='Confusion Matrix Logistic Regression on test data set without features')

In [ ]:
#Traning a Dummy Classifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import log_loss

dummy_clf = DummyClassifier(strategy = "most_frequent")
dummy_clf.fit(X_train,y_train)
score_dummy = dummy_clf.score(X_test, y_test)

pred_proba_dummy = dummy_clf.predict(X_test)
log_loss_dummy = log_loss(y_test,pred_proba_dummy)

print("Testing dummy Acc:",score_dummy )
print("Log loss:",log_loss_dummy )

In [ ]:
#Final Model testing for Unseen data(X_valid, y_valid)
log_reg2 = logistic.train(X_train, y_train, c_val)
prediction_us = log_reg2.predict(X_valid)
report_performance(log_reg2, y_valid, prediction_us, X_valid, y_valid,
                   title='Confusion Matrix Logistic Regression on Validation Data')

## Logistic Regression (with priority-system features)

In [ ]:
#Logistic Regression with features (Priority system)
X_features, y_features = data_split.make_xy(combined_data, use_priority_features=True)
X_train_features, X_valid_features, X_test_features, y_train_features, y_valid_features, y_test_features = \
    data_split.split_train_valid_test(X_features, y_features)

In [ ]:
# C value parameter
df_outcome_features = logistic.sweep_c(X_train_features, y_train_features, X_test_features, y_test_features)
df_outcome_features

In [ ]:
#Training
c_val_features = logistic.best_c(df_outcome_features)
logisticRegr_features = logistic.train(X_train_features, y_train_features, c_val_features)
logisticRegr_features.get_params()

In [ ]:
#Predicting
prediction_features = logistic.summarize_predictions(logisticRegr_features, X_test_features, y_test_features)

In [ ]:
#Confusion Matrix for data with features
report_performance(logisticRegr_features, y_test_features, prediction_features,
                   X_test_features, y_test_features,
                   title='Confusion Matrix Logistic Regression on Test data set With Features')

In [ ]:
#Final Model testing for Unseen data(X_valid, y_valid) with features
log_reg2_features = logistic.train(X_train_features, y_train_features, c_val_features)
prediction_us_features = log_reg2_features.predict(X_valid_features)
report_performance(log_reg2_features, y_valid_features, prediction_us_features,
                   X_valid_features, y_valid_features,
                   title='Confusion Matrix Logistic Regression on Validation Data with Features')

## CatBoost

In [ ]:
#CatBoost library
!pip install catboost

In [ ]:
#Cat Boost on test data (no features)
cbr = catboost_model.train(X_train, y_train)
prediction_cat = cbr.predict(X_test)
report_performance(cbr, y_test, prediction_cat, X_test, y_test,
                   title='Confusion matrix CatBoost on Test Data Set no Features')

In [ ]:
#Cat Boost unseen no features
cbr_us = catboost_model.train(X_train, y_train)
prediction_us_cat = cbr_us.predict(X_valid)
report_performance(cbr_us, y_valid, prediction_us_cat, X_valid, y_valid,
                   title='Confusion Matrix CatBoost on Validation Data Set no Features')

In [ ]:
#Cat Boost with features
cbr_features = catboost_model.train(X_train_features, y_train_features)
prediction_features_cat = cbr_features.predict(X_test_features)
report_performance(cbr_features, y_test_features, prediction_features_cat,
                   X_test_features, y_test_features,
                   title='Confusion Matrix CatBoost on Test Data Set with Features')

In [ ]:
#Cat Boost with features unseen data
cbr_features_us = catboost_model.train(X_train_features, y_train_features)
prediction_features_cat_us = cbr_features_us.predict(X_valid_features)
report_performance(cbr_features_us, y_valid_features, prediction_features_cat_us,
                   X_valid_features, y_valid_features,
                   title='Confusion Matrix CatBoost on Validation Data Set with Features')

## XGBoost

In [ ]:
#Xg Boost on test data (no features)
xgb, prediction_xgb = xgboost_model.train_predict(X_train, y_train, X_test)
report_performance(xgb, y_test, prediction_xgb,
                   X_test._get_numeric_data(), np.ravel(y_test, order='C'),
                   title='Confusion Matrix XGBoost on Test Data Set no Features')

In [ ]:
#Unseen XGBoost
xgb_us, prediction_xgb_us = xgboost_model.train_predict(X_valid, y_valid, X_valid)
report_performance(xgb_us, y_valid, prediction_xgb_us,
                   X_valid._get_numeric_data(), np.ravel(y_valid, order='C'),
                   title='Confusion Matrix XGBoost on Validation Data Set no Features')

In [ ]:
#Xg Boost with features
xgb_features, prediction_xgb_features = xgboost_model.train_predict(
    X_train_features, y_train_features, X_test_features)
report_performance(xgb_features, y_test_features, prediction_xgb_features,
                   X_test_features._get_numeric_data(), np.ravel(y_test_features, order='C'),
                   title='Confusion Matrix XGBoost on Test Data Set with Features')

In [ ]:
#Xg Boost with features Unseen
xgb_features_us, prediction_xgb_features_us = xgboost_model.train_predict(
    X_valid_features, y_valid_features, X_valid_features)
report_performance(xgb_features_us, y_valid_features, prediction_xgb_features_us,
                   X_valid_features._get_numeric_data(), np.ravel(y_valid_features, order='C'),
                   title='Confusion Matrix XGBoost on Validation Data Set with Features')

## AutoGluon Tabular (recommended buy quantity)

In [ ]:
#Importing Library
!pip install autogluon

In [ ]:
#Editing dataset + train test split
combined_data = autogluon_model.add_quantity_targets(combined_data)
X_train_predict, X_valid_predict, X_test_predict, y_train_predict, y_valid_predict, y_test_predict = \
    autogluon_model.split_for_quantity(combined_data)

In [ ]:
#Hyper Parameter for Tabular Predictor + fit
predictor_qty = autogluon_model.train(X_train_predict, X_valid_predict)
performance = predictor_qty.evaluate(X_test_predict)
predictor_qty.leaderboard(X_test_predict, extra_info=True, silent=True)

In [ ]:
#Performance
results = predictor_qty.fit_summary(show_plot=True)
y_test_nolabel = X_test_predict.drop(columns='qty_buy')
y_pred = predictor_qty.predict(y_test_nolabel)
print("Predictions:  ", list(y_pred)[:5])

In [ ]:
#Feature Importance
predictor_qty.feature_importance(X_test_predict)

In [ ]:
#Showing dataset + computing range of error
y_show = autogluon_model.error_report(predictor_qty, X_test_predict, y_test_predict.assign(
    qty_buy=combined_data.loc[y_test_predict.index, 'qty_buy']))